## Задача 4. Split.

Реализуйте разбиение датасета на train, test и val при помощи pandas и без использования циклов на Python. Разбиение должно быть стратифицировано по колонкам, данные должны быть перемешаны. Подробно объясните и/или прокомментируйте, почему ваш код делает то, что нужно.

In [1]:
import pandas
df = pandas.read_csv("estonia-passenger-list.csv")
df

,PassengerId,Country,Firstname,Lastname,Sex,Age,Category,Survived
0,1,Sweden,ARVID KALLE,AADLI,M,62,P,0
1,2,Estonia,LEA,AALISTE,F,22,C,0
2,3,Estonia,AIRI,AAVASTE,F,21,C,0
3,4,Sweden,JURI,AAVIK,M,53,C,0
4,5,Sweden,BRITTA ELISABET,AHLSTROM,F,55,P,0
...,...,...,...,...,...,...,...,...
984,985,Sweden,ANNA INGRID BIRGITTA,OSTROM,F,60,P,0
985,986,Sweden,ELMAR MIKAEL,OUN,M,34,P,1
986,987,Sweden,ENN,QUNAPUU,M,77,P,0
987,988,Sweden,LY,GUNAPUU,F,87,P,0


In [2]:
def split_stratified(df, stratify_columns, train_frac=0.6, val_frac=0.2):
    columns = df.columns

    # Shuffle the data
    df = df.sample(frac = 1, random_state = 42).reset_index(drop = True)
    
    # Group by stratify_columns
    df = df.groupby(stratify_columns, group_keys = False)
    
    # Split each group separately into train/val/test.
    # Add the '__split__' column to each group to specify where each record of each group should go.
    def split_group(group):
        train = int(len(group) * train_frac)     # Last index of train
        val = train + int(len(group) * val_frac) # Last index of val
        group['__split__'] = (
            ['train'] * train +            # Train records
            ['val']   * (val - train) +    # Val records
            ['test']  * (len(group) - val) # All that remains is test
        )
        return group
    df = df[columns].apply(split_group) # After 'apply' on DataFrameGroupBy, df actually becomes a simple DataFrame
    
    # Collect the results into separate data frames and drop the technical '__split__' column.
    train = df[df['__split__'] == 'train'].drop('__split__', axis = 1)
    val   = df[df['__split__'] == 'val'  ].drop('__split__', axis = 1)
    test  = df[df['__split__'] == 'test' ].drop('__split__', axis = 1)
    
    # Cleanup and return
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

stratify_columns = ["Category", "Survived"]
train, val, test = split_stratified(df, stratify_columns)

print(f'Train: {train.shape}')
print(f'Val: {val.shape}')
print(f'Test: {test.shape}')

print(f'Startification in train')
display(train.groupby(stratify_columns).size())

print(f'Startification in val')
display(val.groupby(stratify_columns).size())

print(f'Startification in test')
display(test.groupby(stratify_columns).size())

Train: (591, 8)
Val: (195, 8)
Test: (203, 8)
Startification in train


Category  Survived
C         0            92
          1            23
P         0           418
          1            58
dtype: int64

Startification in val


Category  Survived
C         0            30
          1             7
P         0           139
          1            19
dtype: int64

Startification in test


Category  Survived
C         0            32
          1             9
P         0           141
          1            21
dtype: int64